# Phase 4: Generate Dashboard JSON Payload
This notebook loads the trained models and combined patient data to export a single JSON file for the React batch-inference dashboard.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import shap
import json
import os

base_path = r'C:\Users\thiranbarath\Documents\GitHub\project-d'
ml_data_path = os.path.join(base_path, 'machineLearning', 'csv', 'ml_ready_dataset.csv')
patient_data_path = os.path.join(base_path, 'dataPreprocessing', 'csv', 'step4_patient_aggregated.xlsx')
model_dir = os.path.join(base_path, 'machineLearning', 'models')

print('Loading data...')
df_ml = pd.read_csv(ml_data_path)
df_patient = pd.read_excel(patient_data_path)
print(f'Data loaded: ML shape={df_ml.shape}, Patient shape={df_patient.shape}')


### Loading Models...

This chunk executes the relevant logic for the step mentioned above.

In [ ]:
print('Loading Models...')
xgb_admission = joblib.load(os.path.join(model_dir, 'xgboost_admission.pkl'))
scaler_admission = joblib.load(os.path.join(model_dir, 'standard_scaler.pkl'))

xgb_readmission = joblib.load(os.path.join(model_dir, 'xgboost_readmission.pkl'))
scaler_readmission = joblib.load(os.path.join(model_dir, 'standard_scaler_readmission.pkl'))
thresh_adm = joblib.load(os.path.join(model_dir, 'xgboost_admission_threshold.pkl'))
thresh_readm = joblib.load(os.path.join(model_dir, 'xgboost_readmission_threshold.pkl'))
print('Models and Thresholds loaded successfully.')


### Processing Admission Predictions...

This chunk executes the relevant logic for the step mentioned above.

In [ ]:
print('Processing Admission Predictions...')

leakage_cols_adm = ['Readmitted_Yes_No', 'Num_Admissions', 'Avg_LOS', 'Admitted_Yes_No']
X_adm_raw = df_ml.drop(columns=[c for c in leakage_cols_adm if c in df_ml.columns], errors='ignore')

continuous_cols_adm = ['AGE', 'Num_Visits', 'Total_Meds_Count', 'Total_Unique_Diagnoses', 'Severity_Encoded']
binary_cols_adm = [c for c in X_adm_raw.columns if c not in continuous_cols_adm]
X_adm_scaled = scaler_admission.transform(X_adm_raw)
X_adm_final = pd.DataFrame(X_adm_scaled, columns=continuous_cols_adm + binary_cols_adm, index=X_adm_raw.index)

adm_risk = xgb_admission.predict_proba(X_adm_final)[:, 1]
adm_pred = (adm_risk >= thresh_adm).astype(int)


### Processing Readmission Predictions...

This chunk executes the relevant logic for the step mentioned above.

In [ ]:
print('Processing Readmission Predictions...')

leakage_cols_readm = ['Admitted_Yes_No', 'Num_Admissions', 'Readmitted_Yes_No']
X_readm_raw = df_ml.drop(columns=[c for c in leakage_cols_readm if c in df_ml.columns], errors='ignore')

continuous_cols_readm = ['AGE', 'Avg_LOS', 'Num_Visits', 'Total_Meds_Count', 'Total_Unique_Diagnoses', 'Severity_Encoded']
binary_cols_readm = [c for c in X_readm_raw.columns if c not in continuous_cols_readm]
X_readm_scaled = scaler_readmission.transform(X_readm_raw)
X_readm_final = pd.DataFrame(X_readm_scaled, columns=continuous_cols_readm + binary_cols_readm, index=X_readm_raw.index)

readm_risk = xgb_readmission.predict_proba(X_readm_final)[:, 1]
readm_risk = np.where(adm_pred == 1, readm_risk, np.nan)  # Only compute for predicted admissions


### Calculating Shap Values (Using Subset For Speed Or Treeexplainer On All)...

This chunk executes the relevant logic for the step mentioned above.

In [ ]:
print('Calculating SHAP values (using subset for speed or TreeExplainer on all)...')
explainer_adm = shap.TreeExplainer(xgb_admission)
shap_values_adm = explainer_adm.shap_values(X_adm_final)
explainer_readm = shap.TreeExplainer(xgb_readmission)
shap_values_readm = explainer_readm.shap_values(X_readm_final)

def get_top_features(patient_idx, shap_vals, feature_names):
    # Get base contribution logic
    contributions = shap_vals[patient_idx]
    # Sort by positive contribution to risk
    top_idx = np.argsort(contributions)[-3:][::-1]
    top_features = []
    for i in top_idx:
        if contributions[i] > 0:
            feature_name = feature_names[i]
            impact = (contributions[i] / np.abs(contributions).sum() + 1e-9) * 100
            top_features.append(f'{feature_name} (+{impact:.1f}%)')
    return top_features


### Assembling Json Payload...

This chunk executes the relevant logic for the step mentioned above.

In [ ]:
print('Assembling JSON payload...')
payload = []
feature_names = X_adm_final.columns.tolist()

for idx in range(len(df_patient)):
    pid = df_patient.loc[idx, 'Patient_ID']
    age = int(df_patient.loc[idx, 'AGE'])
    sex = str(df_patient.loc[idx, 'Gender']) if 'Gender' in df_patient.columns else 'U'
    
    if adm_risk[idx] > 0.8:
        severity = 'Critical'
    elif adm_risk[idx] > 0.4:
        severity = 'High'
    elif adm_risk[idx] > 0.15:
        severity = 'Medium'
    else:
        severity = 'Low'
        
    if adm_pred[idx] == 1:
        # If admitted, show drivers for readmission (Stage 2)
        top_drivers = get_top_features(idx, shap_values_readm, list(X_readm_final.columns))
    else:
        # If not admitted, show drivers for admission (Stage 1)
        top_drivers = get_top_features(idx, shap_values_adm, feature_names)
    
    record = {
        'Patient_ID': str(pid),
        'Age': age,
        'Sex': sex,
        'Severity': severity,
        'Stage_1_Admission_Risk': float(adm_risk[idx]),
        'Predicted_Admission': int(adm_pred[idx]),
        'Stage_2_Readmission_Risk': float(readm_risk[idx]) if not np.isnan(readm_risk[idx]) else None,
        'Top_Risk_Drivers': top_drivers
    }
    payload.append(record)

# Export downsampled list if it's too large for dashboard dev (e.g. top 1000 highest risk)
payload.sort(key=lambda x: x['Stage_1_Admission_Risk'], reverse=True)

out_file = os.path.join(base_path, 'diabetesDashboard', 'public', 'data', 'dashboard_payload.json')
os.makedirs(os.path.dirname(out_file), exist_ok=True)
with open(out_file, 'w') as f:
    json.dump(payload, f, indent=2)
print(f'✅ Exported {len(payload)} JSON records to {out_file}')
